# Cuadernillo: implementando VQE

**Joaquín Molina** — Taller de bienvenida, Grupo MoleQLs

Este cuadernillo acompaña la clase teórica ("Fundamento Teórico detrás del VQE"). Vamos a ir revisando, en Qiskit, las mismas piezas que vimos en la teoría: el Hamiltoniano, el principio variacional y el ansatz parametrizado.

Usamos Qiskit porque es el framework más difundido para esto, pero no es la única opción: existen otras plataformas (por ejemplo PennyLane o Cirq), y de hecho nada de lo que sigue depende esencialmente de Qiskit, sin embargp todo podría programarse con Python "nativo" (`numpy` puro), representando estados como vectores y compuertas como matrices. Qiskit simplemente nos da abstracciones cómodas (circuitos, `Statevector`, `SparsePauliOp`) para no reescribir esa álgebra lineal a mano.

In [ ]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit.visualization import array_to_latex


Las celdas de código se apoyan en las anteriores, asi que se recomienda ir en orden y no intentar saltarse celdas.

## Construcción del Hamiltoniano en Qiskit

En la teoría vimos el ejemplo $\hat H = Z$ (Pauli-Z), con autovalores $E=+1$ y $E=-1$, y estado fundamental $|\phi_0\rangle = |1\rangle$ con $E_0=-1$.

Esta vez lo vamos a representar como lo hace Qiskit: con un `SparsePauliOp`, la forma estándar de escribir operadores como combinaciones de operadores de Pauli ($I,X,Y,Z$).

In [ ]:
H = SparsePauliOp(["Z"], coeffs=[1.0])

print("Hamiltoniano H:")
print(H)
print()
print("Representación matricial:")
array_to_latex(H.to_matrix(), prefix="H = ")

Qiskit no diagonaliza el Hamiltoniano por nosotros, pero ya tenemos su matriz. Diagonalicémosla con `numpy` para obtener el espectro completo, tal como en la teoría.

In [ ]:
autovalores, autovectores = np.linalg.eigh(H.to_matrix())

print("Autovalores (energías posibles):", autovalores)

E0 = autovalores[0]
print("Energía del estado fundamental E0:", E0)

### Construyendo los autoestados como circuitos

Ya sabemos por la matemática que $E_0=-1$ y que el estado fundamental es $|\phi_0\rangle=|1\rangle$. Ahora construyamos ese mismo estado con un **circuito cuántico**, y comprobemos con Qiskit que efectivamente tiene esa energía.

Un circuito vacío (sin compuertas) prepara $|0\rangle$. Para preparar $|1\rangle$ basta aplicar una compuerta $X$.

In [ ]:
circuito_0 = QuantumCircuit(1)  # prepara |0>, no le agregamos nada

circuito_1 = QuantumCircuit(1)  # prepara |1>
circuito_1.x(0)

display(circuito_0.draw("mpl"))
display(circuito_1.draw("mpl"))

Para verificar la energía de cada circuito, obtenemos su `Statevector` (el vector de estado que prepara) y calculamos $\langle\psi|\hat H|\psi\rangle$ con `expectation_value`, el mismo valor esperado que definimos en la teoría.

In [ ]:
estado_0 = Statevector.from_instruction(circuito_0)
estado_1 = Statevector.from_instruction(circuito_1)

energia_0 = estado_0.expectation_value(H).real
energia_1 = estado_1.expectation_value(H).real

print("Energía del circuito que prepara |0>:", energia_0)
print("Energía del circuito que prepara |1>:", energia_1)

## Un Hamiltoniano no trivial para VQE

El ejemplo anterior ($\hat H = Z$) es demasiado fácil: su estado fundamental es $|1\rangle$, uno de los dos estados de la base computacional, así que ni siquiera necesitábamos un circuito parametrizado para encontrarlo.

Para que VQE tenga algo interesante que buscar, necesitamos un Hamiltoniano cuyo estado fundamental **no** sea un estado de la base computacional. Vamos a usar:
$$\hat H = 0.5\, Z + 0.5\, X$$

**Tu turno:** completa la celda siguiente construyendo `H` como `SparsePauliOp`, dejando los coeficientes explícitos igual que hicimos arriba con $\hat H = Z$.

In [ ]:
H = SparsePauliOp(["Z","X"], coeffs=[0.5, 0.5])

print("Hamiltoniano H:")
print(H)
print()
print("Representación matricial:")
array_to_latex(H.to_matrix(), prefix="H = ")

Como siempre, diagonalizamos para conocer el espectro exacto. Esto es nuestro "libro de respuestas": lo usaremos más adelante para comprobar si VQE realmente encuentra $E_0$.

In [ ]:
autovalores, autovectores = np.linalg.eigh(H.to_matrix())

E0 = autovalores[0]
phi0 = autovectores[:, 0]

print("Autovalores (espectro):", autovalores)
print("Energía del estado fundamental E0:", E0)
print()
print("Estado fundamental:")
array_to_latex(phi0.reshape(-1, 1), prefix=r"|\phi_0\rangle = ")

Fíjate que `phi0` **no** es $(1,0)$ ni $(0,1)$: el estado fundamental está en superposición. Ningún circuito construido solo con compuertas $X$ (como en la sección anterior) podría prepararlo, por eso ahora sí necesitamos un circuito **parametrizado**.

## El ansatz: parametrizando el estado

Vamos a construir $|\psi(\theta)\rangle$ en dos etapas, tal como se describe en la teoría:

1. **Estado inicial**: en vez de partir de $|0\rangle$, partimos de la superposición uniforme $|+\rangle$, aplicando una compuerta Hadamard.
2. **Compuerta parametrizada**: aplicamos una única rotación $R_y(\theta) = e^{-i\theta Y/2}$, cuyo generador $Y$ es Hermítico (como debe ser).

$$|\psi(\theta)\rangle = R_y(\theta)\, H \, |0\rangle$$

In [ ]:
def crear_ansatz(theta):
    circuito = QuantumCircuit(1)
    circuito.h(0)          # estado inicial: superposición uniforme |+>
    circuito.ry(theta, 0)  # única compuerta parametrizada
    return circuito

crear_ansatz(0.7).draw("mpl")

Con el circuito armado, definimos la función de energía $E(\theta) = \langle\psi(\theta)|\hat H|\psi(\theta)\rangle$, exactamente como en la sección de minimización de la teoría.

In [ ]:
def energia(theta):
    circuito = crear_ansatz(theta)
    estado = Statevector.from_instruction(circuito)
    return estado.expectation_value(H).real

# Probemos con un theta cualquiera
print("E(0.7) =", energia(0.7))

Por el principio variacional, $E(\theta) \geq E_0$ para todo $\theta$. Grafiquemos $E(\theta)$ barriendo muchos valores, y comparémosla con $E_0$ exacto.

In [ ]:
import matplotlib.pyplot as plt

thetas = np.linspace(0, 2 * np.pi, 200)
energias = [energia(t) for t in thetas]

plt.plot(thetas, energias, label=r"$E(\theta)$")
plt.axhline(E0, color="black", linestyle="--", label=r"$E_0$ exacto")
plt.xlabel(r"$\theta$")
plt.ylabel("Energía")
plt.legend()
plt.show()

Busquemos el mínimo directamente sobre los valores calculados, y comparémoslo con $E_0$.

In [ ]:
idx_min = np.argmin(energias)

print("Mínimo encontrado en theta =", thetas[idx_min])
print("Energía mínima encontrada:", energias[idx_min])
print("E0 exacto:", E0)
print("¿Coinciden?", np.isclose(energias[idx_min], E0, atol=1e-2))

Con una sola compuerta parametrizada llegamos exactamente a $E_0$. Esto no es casualidad: nuestro $\hat H = 0.5Z + 0.5X$ no tiene componente en $Y$, así que su estado fundamental queda alcanzable rotando solo con $R_y$ desde $|+\rangle$.

Esto no siempre va a pasar. Más adelante usaremos un Hamiltoniano y un ansatz donde **una sola compuerta no basta**, y necesitaremos combinar varias compuertas con varios parámetros.

## Más parámetros: por qué necesitamos un optimizador

El ansatz anterior funcionó con un solo parámetro porque $\hat H=0.5Z+0.5X$ no tenía componente $Y$. Si agregamos esa componente, el estado fundamental deja de estar en el plano que $R_y$ puede alcanzar desde $|+\rangle$, y un solo parámetro ya no basta. Probemos con:
$$\hat H = 0.5\,X + 0.3\,Y + 0.4\,Z$$

In [ ]:
H = SparsePauliOp(["X", "Y", "Z"], coeffs=[0.5, 0.3, 0.4])

autovalores, autovectores = np.linalg.eigh(H.to_matrix())
E0 = autovalores[0]

print("Autovalores (espectro):", autovalores)
print("Energía del estado fundamental E0:", E0)

Repitamos exactamente el mismo ansatz de antes ($R_y(\theta)$ desde $|+\rangle$) y busquemos su mínimo barriendo una grilla, igual que en la sección anterior.

In [ ]:
thetas = np.linspace(0, 2 * np.pi, 200)
energias = [energia(t) for t in thetas]

plt.plot(thetas, energias, label=r"$E(\theta)$")
plt.axhline(E0, color="black", linestyle="--", label=r"$E_0$ exacto")
plt.xlabel(r"$\theta$")
plt.ylabel("Energía")
plt.legend()
plt.show()

idx_min = np.argmin(energias)

print("Mejor energía encontrada:", energias[idx_min])
print("E0 exacto:", E0)
print("¿Coinciden?", np.isclose(energias[idx_min], E0, atol=1e-2))

Por más fina que hagamos la grilla, nunca vamos a llegar a $E_0$: $R_y(\theta)$ desde $|+\rangle$ solo alcanza estados en un plano del espacio de Bloch, y el estado fundamental de este $\hat H$ queda fuera de ese plano. Necesitamos un segundo parámetro.

Agreguemos una rotación $R_z(\varphi)$ después de $R_y(\theta)$, partiendo esta vez de $|0\rangle$:
$$|\psi(\theta,\varphi)\rangle = R_z(\varphi)\, R_y(\theta) \, |0\rangle$$

Con estos dos ángulos ya podemos alcanzar **cualquier** estado de un qubit (son las coordenadas esféricas del espacio de Bloch). Como ahora tenemos 2 parámetros, en vez de graficar barremos con un optimizador clásico real: `scipy.optimize.minimize`. Esta va a ser nuestra herramienta a partir de ahora, en vez de las grillas.

In [ ]:
from scipy.optimize import minimize

def crear_ansatz_2param(params):
    theta, phi = params
    circuito = QuantumCircuit(1)
    circuito.ry(theta, 0)
    circuito.rz(phi, 0)
    return circuito

def energia_2param(params):
    circuito = crear_ansatz_2param(params)
    estado = Statevector.from_instruction(circuito)
    return estado.expectation_value(H).real

resultado = minimize(energia_2param, x0=[0.5, 0.5], method="Nelder-Mead")

print("Mejor energía con 2 parámetros:", resultado.fun)
print("E0 exacto:", E0)
print("¿Coinciden?", np.isclose(resultado.fun, E0, atol=1e-2))

### Escalando el número de parámetros

Con 2 parámetros todavía podríamos, a duras penas, graficar un mapa de calor 2D para visualizar $E(\theta,\varphi)$. Pero los problemas reales tienen muchos qubits, y cada uno puede necesitar sus propios parámetros. Probemos con **2 qubits independientes** (sin ningún término que los acople), cada uno con un Hamiltoniano de un qubit como el de arriba:
$$\hat H = \underbrace{(0.5X_0+0.3Y_0+0.4Z_0)}_{\text{qubit }0} + \underbrace{(0.2X_1+0.6Y_1+0.1Z_1)}_{\text{qubit }1}$$

Como cada qubit por separado necesita 2 parámetros ($R_y$ y $R_z$), en total tenemos **4 parámetros**. Antes de resolverlo, veamos qué tan cara sería una grilla con esos 4 parámetros (o más):

In [ ]:
puntos_por_parametro = 50

for d in [1, 2, 3, 4, 5, 6]:
    print(f"{d} parámetro(s): {puntos_por_parametro**d:,} evaluaciones para cubrir la grilla completa")

Una grilla se vuelve inviable mucho antes de llegar a los cientos o miles de parámetros que tiene un VQE real. Un optimizador, en cambio, no necesita recorrer toda la grilla: se mueve directamente hacia el mínimo usando la estructura local del problema. Construyamos el Hamiltoniano de 2 qubits y resolvámoslo con `minimize`.

In [ ]:
H = SparsePauliOp(
    ["IX", "IY", "IZ", "XI", "YI", "ZI"],
    coeffs=[0.5, 0.3, 0.4, 0.2, 0.6, 0.1],
)

autovalores, autovectores = np.linalg.eigh(H.to_matrix())
E0 = autovalores[0]

print("Energía del estado fundamental E0:", E0)

In [ ]:
def crear_ansatz_4param(params):
    theta0, phi0, theta1, phi1 = params
    circuito = QuantumCircuit(2)
    circuito.ry(theta0, 0)
    circuito.rz(phi0, 0)
    circuito.ry(theta1, 1)
    circuito.rz(phi1, 1)
    return circuito

def energia_4param(params):
    circuito = crear_ansatz_4param(params)
    estado = Statevector.from_instruction(circuito)
    return estado.expectation_value(H).real

resultado = minimize(energia_4param, x0=[0.1, 0.2, 0.3, 0.4], method="Nelder-Mead")

print("Mejor energía con 4 parámetros:", resultado.fun)
print("E0 exacto:", E0)
print("¿Coinciden?", np.isclose(resultado.fun, E0, atol=1e-2))

El optimizador encuentra $E_0$ sin problema, y la estrategia (llamar a `minimize` una vez) no cambió al pasar de 2 a 4 parámetros (a diferencia de la grilla, que hubiera necesitado millones de evaluaciones extra).

Nota algo importante: este $\hat H$ no tenía **ningún** término que acoplara los dos qubits, así que su estado fundamental sigue siendo un estado producto $|\phi_0^{(0)}\rangle\otimes|\phi_0^{(1)}\rangle$, y por eso un ansatz sin entrelazar (solo $R_y,R_z$ en cada qubit por separado) fue suficiente. La pregunta que queda es: ¿qué pasa si el Hamiltoniano sí acopla los qubits?

## Cuando ni siquiera muchos parámetros bastan: entrelazamiento

En la sección anterior, agregar parámetros alcanzó porque el Hamiltoniano no acoplaba los qubits. Ahora probemos uno que sí los acopla — el Hamiltoniano de intercambio (tipo Heisenberg):
$$\hat H = X\otimes X + Y\otimes Y + Z\otimes Z$$

Vamos a ver que, por más parámetros que le demos a un ansatz sin entrelazar, nunca va a alcanzar $E_0$: hace falta una compuerta que conecte los qubits, como CNOT.

In [ ]:
H = SparsePauliOp(["XX", "YY", "ZZ"], coeffs=[1.0, 1.0, 1.0])

print("Hamiltoniano H:")
print(H)
print()
print("Representación matricial:")
array_to_latex(H.to_matrix(), prefix="H = ")

Como siempre, diagonalizamos para conocer el espectro exacto y el estado fundamental.

In [ ]:
autovalores, autovectores = np.linalg.eigh(H.to_matrix())

E0 = autovalores[0]
phi0 = autovectores[:, 0]

print("Autovalores (espectro):", autovalores)
print("Energía del estado fundamental E0:", E0)
print()
print("Estado fundamental:")
array_to_latex(phi0.reshape(-1, 1), prefix=r"|\phi_0\rangle = ")

$E_0=-3$ y el estado fundamental es el **singlete** $|\phi_0\rangle = \frac{1}{\sqrt2}\big(|01\rangle - |10\rangle\big)$. Este estado no se puede escribir como un producto $|a\rangle\otimes|b\rangle$ de un estado de cada qubit por separado: está **entrelazado**. Por lo tanto ningún ansatz que aplique compuertas de un solo qubit por separado (sin ninguna compuerta que conecte ambos qubits) podrá jamás alcanzarlo, sin importar cuántos parámetros usemos.

### Comprobando que un ansatz sin entrelazar no alcanza

Démosle al ansatz producto todos los parámetros que quiera: $R_y(\theta_0), R_z(\varphi_0)$ en el qubit 0 y $R_y(\theta_1), R_z(\varphi_1)$ en el qubit 1 (4 parámetros, los mismos que nos alcanzaron en la sección anterior), pero **sin ninguna compuerta que los conecte**.

In [ ]:
def crear_ansatz_producto(params):
    theta0, phi0, theta1, phi1 = params
    circuito = QuantumCircuit(2)
    circuito.ry(theta0, 0)
    circuito.rz(phi0, 0)
    circuito.ry(theta1, 1)
    circuito.rz(phi1, 1)
    return circuito

def energia_producto(params):
    circuito = crear_ansatz_producto(params)
    estado = Statevector.from_instruction(circuito)
    return estado.expectation_value(H).real

resultado_producto = minimize(energia_producto, x0=[0.1, 0.2, 0.3, 0.4], method="Nelder-Mead")

print("Mejor energía sin entrelazar (4 parámetros):", resultado_producto.fun)
print("E0 exacto:", E0)
print("¿Coinciden?", np.isclose(resultado_producto.fun, E0, atol=1e-2))

### Agregando una compuerta que entrelace

Por más que optimicemos los 4 parámetros, el ansatz producto se estanca lejos de $E_0=-3$: el problema no es la cantidad de parámetros, sino que ningún estado producto puede representar un estado entrelazado. Agreguemos una compuerta CNOT después de las rotaciones para permitir que los dos qubits queden entrelazados:
$$|\psi(\vec\theta)\rangle = \text{CNOT}_{0,1}\; \big(R_z(\varphi_0)R_y(\theta_0)\otimes R_z(\varphi_1)R_y(\theta_1)\big) \, |00\rangle$$

In [ ]:
def crear_ansatz_entrelazado(params):
    theta0, phi0, theta1, phi1 = params
    circuito = QuantumCircuit(2)
    circuito.ry(theta0, 0)
    circuito.rz(phi0, 0)
    circuito.ry(theta1, 1)
    circuito.rz(phi1, 1)
    circuito.cx(0, 1)  # compuerta que entrelaza los dos qubits
    return circuito

crear_ansatz_entrelazado([0.1, 0.2, 0.3, 0.4]).draw("mpl")

In [ ]:
def energia_entrelazada(params):
    circuito = crear_ansatz_entrelazado(params)
    estado = Statevector.from_instruction(circuito)
    return estado.expectation_value(H).real

resultado = minimize(energia_entrelazada, x0=[0.1, 0.2, 0.3, 0.4], method="Nelder-Mead")

print("Mejor energía con CNOT:", resultado.fun)
print("E0 exacto:", E0)
print("¿Coinciden?", np.isclose(resultado.fun, E0, atol=1e-2))

Con la CNOT agregada, el optimizador sí encuentra $E_0=-3$: esta es la idea central de un ansatz para VQE en varios qubits — combinar rotaciones parametrizadas de un qubit con compuertas que entrelacen, como CNOT, para que la familia de estados de prueba pueda alcanzar estados entrelazados como el estado fundamental real.

## Entonces, ¿cómo sé qué ansatz usar en cada problema?

Con lo visto hasta ahora, la receta parece simple: si el ansatz no alcanza, agregar parámetros; si tampoco alcanza, agregar compuertas que entrelacen. En la práctica no es tan directo, porque agregar parámetros sin límite trae sus propios problemas.

**Sobreparametrización.** Más parámetros no es gratis. Cada parámetro extra es una dimensión más que el optimizador clásico tiene que explorar: la optimización se vuelve más lenta, aparecen más mínimos locales, y cada evaluación de $E(\vec\theta)$ (en hardware real, con ruido) cuesta más. Si el ansatz ya podía representar el estado fundamental con pocos parámetros, agregar más no mejora la precisión, solo encarece el entrenamiento. Elegir un ansatz demasiado grande "por si acaso" tiene un costo real.

Por eso, en la práctica casi nunca se usa un ansatz genérico como los nuestros para cualquier problema: se elige (o se diseña) un ansatz **adaptado a la estructura física del problema**. Algunos ejemplos:

- **Hardware-efficient ansatz**: el tipo que construimos en este cuadernillo (capas de rotaciones de un qubit + compuertas de entrelazamiento, repetidas). No usa nada de la física del problema, solo lo que es barato de ejecutar en el hardware disponible. Es muy flexible, pero al no estar guiado por el problema puede necesitar más parámetros o profundidad de la estrictamente necesaria.

- **UCCSD** (*Unitary Coupled Cluster Singles and Doubles*): el ansatz típico en química cuántica para moléculas. Parte del estado de Hartree-Fock (la mejor aproximación clásica de campo medio) y aplica un unitario generado por **excitaciones de uno y dos electrones** desde los orbitales ocupados hacia los virtuales. Al estar basado en la física del problema (conserva el número de partículas, y se inspira en un desarrollo ya usado en química clásica), suele necesitar menos parámetros "desperdiciados" que un ansatz genérico pero a costa de circuitos típicamente más profundos.

- **ADAPT-VQE**: un punto intermedio entre los dos anteriores. En vez de fijar el ansatz completo de antemano, se construye **adaptativamente**: en cada paso se agrega, de un conjunto de operadores candidatos (como las excitaciones de UCCSD), el que más reduce la energía. Así el tamaño del ansatz se ajusta al problema, en vez de sobre- o sub-parametrizar desde el principio.

No hay un ansatz universal: elegir (o construir) el correcto para tu problema, usando tanto como sea posible de su estructura física, sin agregar parámetros de más sigue siendo un área activa de investigación en computación cuántica. 



> **Counterdiabatic ADAPT-VQE for Molecular Simulation**
> Diego Tancara, Herbert Díaz-Moraga, Dardo Goyeneche
> *Journal of Chemical Theory and Computation*, 2026, 22(13), 6419–6430
> DOI: [10.1021/acs.jctc.6c00197](https://doi.org/10.1021/acs.jctc.6c00197)
